# Gold Phase: Risk Scoring, Metrics & KPI Mart
Unified notebook combining: Risk Scoring → Rolling Metrics → KPI Mart

## Initialize & Load Silver Data

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from datetime import datetime

spark = SparkSession.builder.appName("gold-complete").getOrCreate()

print("\n=== GOLD LAYER - COMPLETE ANALYTICS ===")
print(f"Start time: {datetime.now()}")

# Load silver data
sdf = spark.read.table("data_engineering_workshop.creditcard.creditcard_silver_final")
print(f"\nLoaded silver data: {sdf.count():,} rows")

## Step 1: Fraud Risk Scoring

In [ ]:
print(f"\n=== STEP 1: FRAUD RISK SCORING ===")

# Build risk score model
sdf = sdf.withColumn(
    "fraud_risk_score",
    (
        (F.when(F.col("amount_normalized") > 0.5, 0.4).otherwise(0.1)) +
        (F.when(F.col("is_night_hour"), 0.2).otherwise(0.0)) +
        (F.when(F.abs(F.col("v18")) > 3.0, 0.3).otherwise(0.0)) +
        (F.when(F.abs(F.col("v19")) > 3.0, 0.2).otherwise(0.0)) +
        (F.when(F.abs(F.col("v20")) > 3.0, 0.2).otherwise(0.0)) +
        (F.when(F.col("is_proxy_duplicate"), 0.1).otherwise(0.0)) +
        (F.when(F.col("is_high_amount"), 0.15).otherwise(0.0)) +
        (F.when(
            (F.col("is_night_hour")) & (F.col("is_high_amount")),
            0.2
        ).otherwise(0.0))
    )
)

# Normalize to 0-100 scale
max_score = 2.0
sdf = sdf.withColumn(
    "fraud_risk_score_pct",
    F.when(
        (F.col("fraud_risk_score") / max_score * 100).cast("int") > 100, 100
    ).otherwise((F.col("fraud_risk_score") / max_score * 100).cast("int"))
)

# Risk tiers
sdf = sdf.withColumn(
    "risk_tier",
    F.when(F.col("fraud_risk_score_pct") >= 75, "CRITICAL")
     .when(F.col("fraud_risk_score_pct") >= 50, "HIGH")
     .when(F.col("fraud_risk_score_pct") >= 25, "MEDIUM")
     .otherwise("LOW")
)

# Recommended actions
sdf = sdf.withColumn(
    "recommended_action",
    F.when(F.col("risk_tier") == "CRITICAL", "BLOCK_IMMEDIATE")
      .when(F.col("risk_tier") == "HIGH", "REQUEST_VERIFICATION")
      .when(F.col("risk_tier") == "MEDIUM", "FLAG_FOR_REVIEW")
      .otherwise("APPROVE")
)
#
# Prediction confidence
anomaly_count = (
    F.when(F.abs(F.col("v18")) > 2.5, 1).otherwise(0) +
    F.when(F.abs(F.col("v19")) > 2.5, 1).otherwise(0) +
    F.when(F.abs(F.col("v20")) > 2.5, 1).otherwise(0)
)

sdf = sdf.withColumn("anomaly_feature_count", anomaly_count)

sdf = sdf.withColumn(
    "prediction_confidence",
    F.when(F.col("anomaly_feature_count") >= 2, 0.95)
      .when(F.col("anomaly_feature_count") == 1, 0.80)
      .when(F.col("is_high_amount") & F.col("is_night_hour"), 0.85)
      .otherwise(0.70)
)

# Write risk scores
sdf.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_gold_risk_scores"
)
print(f"✓ Risk scores created: {sdf.count():,} rows")

# Show distribution
risk_dist = sdf.groupBy("risk_tier").count().collect()
for row in risk_dist:
    print(f"  {row['risk_tier']}: {row['count']:,}")

## Step 2: Rolling Metrics & Aggregations

In [ ]:
print(f"\n=== STEP 2: ROLLING METRICS ===")

# Load risk scores
risk_df = spark.read.table("data_engineering_workshop.creditcard.creditcard_gold_risk_scores")

# Hourly aggregations
risk_df = risk_df.withColumn(
    "hour_bucket",
    (F.col("time") / 3600).cast("int") * 3600
)

hourly = risk_df.groupBy("hour_bucket").agg(
    F.count("*").alias("total_transactions"),
    F.sum(F.col("class")).alias("fraud_count"),
    F.round(F.sum(F.col("class")) / F.count("*") * 100, 2).alias("fraud_rate_pct"),
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.round(F.sum(F.col("amount")), 2).alias("total_amount")
).orderBy("hour_bucket")

# Daily aggregations
risk_df_daily = risk_df.withColumn(
    "day_bucket",
    (F.col("time") / (24*3600)).cast("int") * (24*3600)
)

daily = risk_df_daily.groupBy("day_bucket").agg(
    F.count("*").alias("total_transactions"),
    F.sum(F.col("class")).alias("fraud_count"),
    F.round(F.sum(F.col("class")) / F.count("*") * 100, 2).alias("fraud_rate_pct"),
    F.round(F.sum(F.col("amount")), 2).alias("total_amount"),
    F.round(F.avg("amount"), 2).alias("avg_amount")
).orderBy("day_bucket")

# Write metrics
hourly.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_gold_hourly_metrics"
)

daily.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_gold_daily_metrics"
)

print(f"✓ Hourly metrics: {hourly.count()} records")
print(f"✓ Daily metrics: {daily.count()} records")

## Step 3: KPI Mart for Dashboards

In [ ]:
print(f"\n=== STEP 3: KPI MART ===")

# Load risk scores again for KPI calculation
risk_df = spark.read.table("data_engineering_workshop.creditcard.creditcard_gold_risk_scores")

# Overall KPIs
overall_metrics = risk_df.agg(
    F.count("*").alias("total_transactions"),
    F.sum(F.col("class")).alias("fraud_cases"),
    F.round(
        F.sum(F.col("class")) / F.count("*") * 100, 3
    ).alias("fraud_rate_pct"),
    F.round(F.sum("amount"), 2).alias("total_transaction_amount"),
    F.round(F.avg("amount"), 2).alias("avg_transaction_amount"),
    F.round(
        F.sum(F.when(F.col("class") == 1, F.col("amount")).otherwise(0)), 2
    ).alias("fraud_total_amount"),
    F.round(F.avg(F.col("fraud_risk_score_pct")), 2).alias("avg_risk_score"),
    F.lit(datetime.now()).cast("timestamp").alias("snapshot_timestamp")
)

# Risk tier KPIs
risk_tier_kpis = risk_df.groupBy("risk_tier").agg(
    F.count("*").alias("transaction_count"),
    F.sum(F.col("class")).alias("fraud_count"),
    F.round(
        F.sum(F.col("class")) / F.count("*") * 100, 2
    ).alias("fraud_rate_pct"),
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.lit(datetime.now()).cast("timestamp").alias("snapshot_timestamp")
)

# Time-based KPIs
risk_df_time = risk_df.withColumn(
    "time_period",
    F.when(F.col("hour_of_day") < 6, "Night (0-6)")
      .when(F.col("hour_of_day") < 12, "Morning (6-12)")
      .when(F.col("hour_of_day") < 18, "Afternoon (12-18)")
      .otherwise("Evening (18-24)")
)

time_kpis = risk_df_time.groupBy("time_period").agg(
    F.count("*").alias("transaction_count"),
    F.sum(F.col("class")).alias("fraud_count"),
    F.round(
        F.sum(F.col("class")) / F.count("*") * 100, 2
    ).alias("fraud_rate_pct"),
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.lit(datetime.now()).cast("timestamp").alias("snapshot_timestamp")
)

# Write KPI mart
overall_metrics.write.mode("overwrite").format("delta").saveAsTable(
    "data_engineering_workshop.creditcard.creditcard_gold_kpi_mart"
)

print(f"✓ Overall KPI Mart created")
overall_metrics.show()

print(f"\nRisk Tier Distribution:")
risk_tier_kpis.show()

print(f"\nTime Period Analysis:")
time_kpis.show()

## Gold Layer Complete Summary

In [ ]:
print(f"\n=== GOLD LAYER COMPLETE ===")
print(f"End time: {datetime.now()}")

print(f"\n✓ Tables created:")
print(f"  - creditcard_gold_risk_scores")
print(f"  - creditcard_gold_hourly_metrics")
print(f"  - creditcard_gold_daily_metrics")
print(f"  - creditcard_gold_kpi_mart")
print(f"\n✓ All analytics ready for dashboards and reporting!")